In [1]:
import pandas as pd
Finaldata = pd.read_csv("/data/ManufactureFinalDatastes.csv")

In [2]:
print (Finaldata)

        Unnamed: 0 EquipmentID         Timestamp  Temperature  Vibration  \
0                0   Equip_007  01-01-2023 00:00     0.682891   0.370087   
1                1   Equip_007  01-01-2023 00:00     0.682891   0.370087   
2                2   Equip_007  01-01-2023 00:00     0.682891   0.370087   
3                3   Equip_007  01-01-2023 00:00     0.682891   0.370087   
4                4   Equip_007  01-01-2023 00:00     0.682891   0.370087   
...            ...         ...               ...          ...        ...   
249858      249858   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249859      249859   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249860      249860   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249861      249861   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249862      249862   Equip_013  28-07-2023 00:00     0.253512   0.964459   

        Pressure  OperationalLoad     RecordID   MaintenanceDate  \
0       0.145886   

In [3]:
Finaldata['time_since_last_maintenance'] = Finaldata['time_since_last_maintenance'].str[:3]

In [4]:
Finaldata['time_since_last_maintenance']

0         499
1         499
2         499
3         499
4         499
         ... 
249858    291
249859    291
249860    291
249861    291
249862    291
Name: time_since_last_maintenance, Length: 249863, dtype: object

In [5]:
from tqdm import tqdm
dtypes = pd.DataFrame(Finaldata.dtypes)
dtypes.reset_index(inplace=True)

In [6]:
for i_ in tqdm(range(len(dtypes))):
    col_name = dtypes["index"].iloc[i_]
    if Finaldata[col_name].isna().sum() != 0:
        if Finaldata[col_name].dtype != "object":
            Finaldata[col_name].fillna(value=Finaldata[col_name].mean(), inplace=True)
        else:
            Finaldata[col_name].fillna(value=Finaldata[col_name].mode()[0], inplace=True)

100%|██████████| 24/24 [00:00<00:00, 255.96it/s]


In [7]:
print (Finaldata)

        Unnamed: 0 EquipmentID         Timestamp  Temperature  Vibration  \
0                0   Equip_007  01-01-2023 00:00     0.682891   0.370087   
1                1   Equip_007  01-01-2023 00:00     0.682891   0.370087   
2                2   Equip_007  01-01-2023 00:00     0.682891   0.370087   
3                3   Equip_007  01-01-2023 00:00     0.682891   0.370087   
4                4   Equip_007  01-01-2023 00:00     0.682891   0.370087   
...            ...         ...               ...          ...        ...   
249858      249858   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249859      249859   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249860      249860   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249861      249861   Equip_013  28-07-2023 00:00     0.253512   0.964459   
249862      249862   Equip_013  28-07-2023 00:00     0.253512   0.964459   

        Pressure  OperationalLoad     RecordID   MaintenanceDate  \
0       0.145886   

In [8]:
features = Finaldata.drop(columns=['EquipmentID', 'Timestamp', 'FailureFlag'])  # Adjust based on your target variable
target = Finaldata['FailureFlag']  # Assuming 'FailureFlag' is your target variable for classification


In [9]:
# X= training columns and Y is target column
Finaldata_x = Finaldata[['Temperature','Vibration','Pressure','OperationalLoad','Temperature_7d_avg',
 'Vibration_7d_avg','Pressure_7d_avg','time_since_last_maintenance','sum','OperationalEfficiency','OperationalLoad_7d_avg' ]]
Finaldata_y = Finaldata['FailureFlag']


In [10]:
Finaldata_y.head()

0    1
1    0
2    0
3    1
4    1
Name: FailureFlag, dtype: int64

In [11]:
X = Finaldata_x
y = Finaldata_y

In [12]:
#Split the Data: Divide the data into training and testing sets to evaluate the model's performance
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Step 2: Initialize the Random Forest Model

In [13]:
#Import the Model: Import the RandomForestRegressor class from scikit-learn for regression tasks.

from sklearn.ensemble import RandomForestRegressor
 

In [14]:
# Create an Instance: Initialize the Random Forest model. You can start with default parameters or specify some based on prior knowledge or heuristics.

Rf_model = RandomForestRegressor(random_state=42)


# Step 3: Train the Model

In [15]:
Rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

# Step 4: Model Evaluation
# Make Predictions: Use the trained model to predict outcomes on the test set.




In [16]:
y_pred = Rf_model.predict(X_test)

In [17]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")
# Assess Performance: Evaluate the model using regression metrics such as Mean Squared Error (MSE) and R-squared.

Mean Squared Error: 0.26444136886491415
R-squared: -0.05780857946295348


# Step 5: Hyperparameter Tuning
#Tune Parameters: Use techniques like grid search to find the best set of hyperparameters for your model. Key parameters for Random Forest include n_estimators, max_depth, min_samples_split, and min_samples_leaf.


In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}



In [ ]:
grid_search = GridSearchCV(Rf_model, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)


# Apply Best Parameters: Use the best parameters from the grid search to configure your Random Forest model

In [ ]:
best_Rf_model = grid_search.best_estimator_